# YOLOv8 ONNX to TensorRT Conversion

This notebook converts YOLOv8 object detection models from ONNX format to TensorRT for optimized inference.

## Prerequisites
- NVIDIA GPU with CUDA support
- TensorRT installed
- Python packages: `tensorrt`, `numpy`

## 1. Import Dependencies and Check Installation

In [1]:
import os
import sys
import numpy as np
from pathlib import Path

# Check TensorRT installation
try:
    import tensorrt as trt
    print(f"✅ TensorRT version: {trt.__version__}")
except ImportError:
    print("❌ Error: TensorRT not found. Please install TensorRT.")
    print("Installation instructions: https://docs.nvidia.com/deeplearning/tensorrt/install-guide/index.html")
    raise

✅ TensorRT version: 10.10.0.31


## 2. Configuration Settings

Modify these variables according to your needs:

In [26]:
# Configuration - modify these variables as needed
onnx_path = r"C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\models\weights\silent_contract_9082.onnx"  # Path to input ONNX model
engine_path = r"C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\models\weights\silent_contract_9082_960px_bs64_trt.engine"  # Path to output TensorRT engine
precision = "fp16"  # Precision mode: 'fp32', 'fp16', or 'int8'
max_batch_size = 256  # Maximum batch size
input_height = 960  # Input height
input_width = 960  # Input width
verbose = True  # Enable verbose logging
verify_engine = True  # Verify engine after conversion

print("Configuration:")
print(f"  ONNX Model: {onnx_path}")
print(f"  Output Engine: {engine_path}")
print(f"  Precision: {precision}")
print(f"  Max Batch Size: {max_batch_size}")
print(f"  Input Size: {input_height}x{input_width}")

Configuration:
  ONNX Model: C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\models\weights\silent_contract_9082.onnx
  Output Engine: C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\models\weights\silent_contract_9082_960px_bs64_trt.engine
  Precision: fp16
  Max Batch Size: 256
  Input Size: 960x960


## 3. YOLOv8 ONNX to TensorRT Converter Class

In [27]:
class YOLOv8ONNXToTensorRT:
    def __init__(self, onnx_path, engine_path, precision='fp16', max_batch_size=1, input_shape=(64, 3, 640, 640), verbose=False):
        """
        Initialize the converter
        
        Args:
            onnx_path (str): Path to input ONNX model
            engine_path (str): Path to output TensorRT engine
            precision (str): Precision mode ('fp32', 'fp16', 'int8')
            max_batch_size (int): Maximum batch size for optimization
            input_shape (tuple): Input tensor shape (batch, channels, height, width)
            verbose (bool): Enable verbose logging
        """
        self.onnx_path = Path(onnx_path)
        self.engine_path = Path(engine_path)
        self.precision = precision.lower()
        self.max_batch_size = max_batch_size
        self.input_shape = input_shape
        self.verbose = verbose
        
        # Validate inputs
        if not self.onnx_path.exists():
            raise FileNotFoundError(f"ONNX file not found: {self.onnx_path}")
        
        if self.precision not in ['fp32', 'fp16', 'int8']:
            raise ValueError(f"Unsupported precision: {self.precision}")
        
        # Create output directory
        self.engine_path.parent.mkdir(parents=True, exist_ok=True)
        
        # Initialize TensorRT logger
        self.logger = trt.Logger(trt.Logger.VERBOSE if verbose else trt.Logger.WARNING)
        
    def build_engine(self):
        """Build TensorRT engine from ONNX model"""
        print(f"🔨 Building TensorRT engine from {self.onnx_path}")
        print(f"📄 Output engine: {self.engine_path}")
        print(f"⚙️  Precision: {self.precision}")
        print(f"📦 Max batch size: {self.max_batch_size}")
        print(f"📐 Input shape: {self.input_shape}")
        
        # Create builder and network
        builder = trt.Builder(self.logger)
        network = builder.create_network(1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH))
        parser = trt.OnnxParser(network, self.logger)
        
        # Parse ONNX model
        print("🔍 Parsing ONNX model...")
        with open(self.onnx_path, 'rb') as model:
            if not parser.parse(model.read()):
                print("❌ Failed to parse ONNX model")
                for error in range(parser.num_errors):
                    print(f"Error {error}: {parser.get_error(error)}")
                return False
        
        print(f"✅ Network inputs: {network.num_inputs}")
        print(f"✅ Network outputs: {network.num_outputs}")
        
        # Print input/output info
        for i in range(network.num_inputs):
            input_tensor = network.get_input(i)
            print(f"📥 Input {i}: {input_tensor.name}, shape: {input_tensor.shape}, dtype: {input_tensor.dtype}")
        
        for i in range(network.num_outputs):
            output_tensor = network.get_output(i)
            print(f"📤 Output {i}: {output_tensor.name}, shape: {output_tensor.shape}, dtype: {output_tensor.dtype}")
        
        # Create builder config
        config = builder.create_builder_config()
        
        # Set precision
        if self.precision == 'fp16':
            if builder.platform_has_fast_fp16:
                config.set_flag(trt.BuilderFlag.FP16)
                print("🚀 FP16 precision enabled")
            else:
                print("⚠️  Warning: FP16 not supported on this platform, using FP32")
        elif self.precision == 'int8':
            if builder.platform_has_fast_int8:
                config.set_flag(trt.BuilderFlag.INT8)
                print("🚀 INT8 precision enabled")
                # Note: For INT8, you would need to provide calibration data
                print("⚠️  Warning: INT8 calibration not implemented in this script")
            else:
                print("⚠️  Warning: INT8 not supported on this platform, using FP32")
        
        # Set optimization profiles for dynamic shapes
        profile = builder.create_optimization_profile()
        
        # Get input tensor name
        input_name = network.get_input(0).name
        
        # Set min, opt, max shapes for dynamic batching
        min_shape = (1, self.input_shape[1], self.input_shape[2], self.input_shape[3])
        opt_shape = self.input_shape
        max_shape = (self.max_batch_size, self.input_shape[1], self.input_shape[2], self.input_shape[3])
        
        profile.set_shape(input_name, min_shape, opt_shape, max_shape)
        config.add_optimization_profile(profile)
        
        print("🔧 Building TensorRT engine... This may take several minutes.")
        
        # Build engine
        try:
            engine = builder.build_serialized_network(network, config)
            if engine is None:
                print("❌ Failed to build TensorRT engine")
                return False
        except Exception as e:
            print(f"❌ Error building engine: {e}")
            return False
        
        # Serialize and save engine
        print(f"💾 Saving engine to {self.engine_path}")
        with open(self.engine_path, 'wb') as f:
            f.write(engine)
        
        print("✅ TensorRT engine conversion completed successfully!")
        print(f"📊 Engine size: {self.engine_path.stat().st_size / (1024*1024):.2f} MB")
        
        return True
    
    def verify_engine(self):
        """Verify the built engine can be loaded"""
        print("🔍 Verifying engine...")
        try:
            runtime = trt.Runtime(self.logger)
            with open(self.engine_path, 'rb') as f:
                engine = runtime.deserialize_cuda_engine(f.read())
            
            if engine is None:
                print("❌ Failed to load engine")
                return False
            
            print(f"✅ Engine verification successful!")
            
            # Check for max_batch_size attribute (removed in newer TensorRT)
            if hasattr(engine, 'max_batch_size'):
                print(f"📦 Max batch size: {engine.max_batch_size}")
            else:
                print("📦 Max batch size: Dynamic (newer TensorRT version)")
                
            print(f"🔗 Number of bindings: {engine.num_bindings}")
            
            # Print binding info with updated API
            for i in range(engine.num_bindings):
                name = engine.get_binding_name(i)
                
                # Handle different API versions for getting shapes
                if hasattr(engine, 'get_binding_shape'):
                    shape = engine.get_binding_shape(i)
                elif hasattr(engine, 'get_tensor_shape'):
                    shape = engine.get_tensor_shape(name)
                else:
                    shape = "Unknown"
                    
                dtype = engine.get_binding_dtype(i)
                is_input = engine.binding_is_input(i)
                print(f"🔗 Binding {i}: {name}, shape: {shape}, dtype: {dtype}, is_input: {is_input}")
            
            return True
        
        except Exception as e:
            print(f"❌ Engine verification failed: {e}")
            return False

print("✅ YOLOv8ONNXToTensorRT class defined successfully!")

✅ YOLOv8ONNXToTensorRT class defined successfully!


## 4. Check Input ONNX Model

In [28]:
# Check if ONNX model exists
onnx_file = Path(onnx_path)
if onnx_file.exists():
    print(f"✅ ONNX model found: {onnx_path}")
    print(f"📊 File size: {onnx_file.stat().st_size / (1024*1024):.2f} MB")
else:
    print(f"❌ ONNX model not found: {onnx_path}")
    print("Please ensure the ONNX model path is correct and the file exists.")

✅ ONNX model found: C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\models\weights\silent_contract_9082.onnx
📊 File size: 98.72 MB


## 5. Create Converter and Build Engine

In [29]:
# Set input shape
input_shape = (max_batch_size, 3, input_height, input_width)

# Create converter
converter = YOLOv8ONNXToTensorRT(
    onnx_path=onnx_path,
    engine_path=engine_path,
    precision=precision,
    max_batch_size=max_batch_size,
    input_shape=input_shape,
    verbose=verbose
)

print("🚀 Starting TensorRT engine conversion...")
print("=" * 50)

🚀 Starting TensorRT engine conversion...


## 6. Build the TensorRT Engine

In [30]:
# Convert model
success = converter.build_engine()

if success:
    print("\n" + "="*50)
    print("✅ Conversion completed successfully!")
    print(f"📄 TensorRT engine saved to: {engine_path}")
else:
    print("\n" + "="*50)
    print("❌ Conversion failed!")
    print("Please check the error messages above and verify your ONNX model.")

🔨 Building TensorRT engine from C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\models\weights\silent_contract_9082.onnx
📄 Output engine: C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\models\weights\silent_contract_9082_960px_bs64_trt.engine
⚙️  Precision: fp16
📦 Max batch size: 256
📐 Input shape: (256, 3, 960, 960)
🔍 Parsing ONNX model...
✅ Network inputs: 1
✅ Network outputs: 1
📥 Input 0: images, shape: (-1, 3, -1, -1), dtype: DataType.FLOAT
📤 Output 0: output0, shape: (-1, 13, -1), dtype: DataType.FLOAT
🚀 FP16 precision enabled
🔧 Building TensorRT engine... This may take several minutes.


❌ Failed to build TensorRT engine

❌ Conversion failed!
Please check the error messages above and verify your ONNX model.


## 7. Verify the Built Engine (Optional)

In [33]:
# Verify engine if conversion was successful and verification is enabled
verify_engine = True
if success and verify_engine:
    print("\n" + "="*30)
    verification_success = converter.verify_engine()
    
    if verification_success:
        print("\n🎉 Engine verification passed! Your TensorRT engine is ready to use.")
    else:
        print("\n⚠️  Engine verification failed. The engine file may be corrupted.")
else:
    print("\n⏭️  Skipping engine verification.")


⏭️  Skipping engine verification.


## 8. Summary and Next Steps

In [32]:
print("\n" + "="*50)
print("📋 CONVERSION SUMMARY")
print("="*50)
print(f"Input ONNX Model: {onnx_path}")
print(f"Output TensorRT Engine: {engine_path}")
print(f"Precision: {precision.upper()}")
print(f"Max Batch Size: {max_batch_size}")
print(f"Input Resolution: {input_height}x{input_width}")

if Path(engine_path).exists():
    engine_size = Path(engine_path).stat().st_size / (1024*1024)
    print(f"Engine Size: {engine_size:.2f} MB")
    print("\n✅ Your TensorRT engine is ready for inference!")
    print("\n📝 Next Steps:")
    print("1. Load the engine in your inference code")
    print("2. Create CUDA context and allocate GPU memory")
    print("3. Run inference with your optimized model")
    print("\n💡 Expected performance improvements:")
    print("- 2-5x faster inference compared to ONNX")
    print("- Lower memory usage")
    print("- Better GPU utilization")
else:
    print("\n❌ Engine file not found. Conversion may have failed.")

print("\n" + "="*50)


📋 CONVERSION SUMMARY
Input ONNX Model: C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\models\weights\silent_contract_9082.onnx
Output TensorRT Engine: C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\models\weights\silent_contract_9082_960px_bs64_trt.engine
Precision: FP16
Max Batch Size: 256
Input Resolution: 960x960
Engine Size: 53.33 MB

✅ Your TensorRT engine is ready for inference!

📝 Next Steps:
1. Load the engine in your inference code
2. Create CUDA context and allocate GPU memory
3. Run inference with your optimized model

💡 Expected performance improvements:
- 2-5x faster inference compared to ONNX
- Lower memory usage
- Better GPU utilization

